## 6. Create the train/validation/test split

This block prepares the graph for the supervised drug repurposing task. The next step is to define exactly what the model should learn, split the target edges into training and evaluation sets, generate negative examples for validation and testing, and package everything into a PyTorch Geometric `HeteroData` object.

### A. Separate target edges from context edges

The block begins by copying `edges_model_ready_df` and splitting it into two parts:

- **target edges**: the edges the model is asked to predict  
- **context edges**: all other edge types that remain in the graph as supporting structure

Here, the target relation is set to:

- `TARGET_EDGE_TRIPLET = ("drug", "indication", "disease")`
- `TARGET_EDGE_LABEL = "indication"`

This means the model is specifically learning to predict whether a **drug** should connect to a **disease** through the **indication** edge type.

The code then checks that all retained indication edges really follow the expected direction of **drug → disease**. This is an important sanity check, because any leftover direction inconsistency would affect both training and evaluation.

After that, the target edges are deduplicated using the `(drug, disease)` pair. This ensures that the same therapeutic link is not counted multiple times in the supervision set.

### B. Split the target edges into train, validation, and test sets

Once the target edges are isolated, the block divides them into three splits:

- **70% training**
- **15% validation**
- **15% test**

The block also checks for leakage by making sure that no `(drug, disease)` pair appears in more than one split. This matters because if the same positive pair were present in both training and evaluation, model performance would be overstated.

### C. Generate negative drug–disease pairs

Link prediction requires both positive and negative examples. The positive examples are the known indication edges from PrimeKG, but the model also needs examples of drug–disease pairs that are **not** known indications.

To do this, the block defines a helper function called `sample_negatives(...)`. This function randomly samples drug–disease index pairs while avoiding any pair that already exists as a known positive indication edge.

A few important details about this step:

- the sampler works over the full range of drug indices and disease indices
- it rejects any sampled pair that is already a known positive
- it also avoids duplicate negative pairs
- it uses a seeded random generator so results are reproducible

The full set of positive indication pairs across all splits is collected into `all_positive_pairs`. This prevents the negative sampler from accidentally drawing a real therapeutic edge.

In this notebook, negative samples are generated for:
- **validation**
- **test**

The evaluation is balanced using:

- `NEG_RATIO = 1`

### D. Build the supervision tensors

After the splits are created, the block converts each supervision dataframe into PyTorch tensors using the helper function `df_to_edge_index(...)`.

Each tensor has shape:

`[2, number_of_edges]`

where:
- the first row contains source node indices
- the second row contains destination node indices

The resulting tensors are:

- `train_pos_edge_label_index`
- `val_pos_edge_label_index`
- `test_pos_edge_label_index`

along with the pre-sampled evaluation negatives:

- `val_neg_edge_label_index`
- `test_neg_edge_label_index`

### E. Build the PyTorch Geometric `HeteroData` object

The final part of the block constructs `train_data`, a PyTorch Geometric `HeteroData` object that stores the heterogeneous graph used during training.

#### Node setup

For each node type, the block stores only:

- `num_nodes`

It intentionally does **not** assign explicit node feature vectors (`.x`) at this stage.

The notebook avoids creating artificial scalar features like node indices, because doing so would impose an arbitrary numeric structure that does not meaningfully represent the biomedical entities.

#### Edge setup

The graph is then built from:
- **all context edges**
- **only the training indication edges**

So the training graph contains:
- every non-target biomedical relation as full context
- only the train portion of the `indication` edges

In [ ]:
# Train / Val / Test Split + PyG HeteroData

assert "edges_model_ready_df" in globals(), "Run Block 04 first."
assert "num_nodes_dict"        in globals(), "Run Block 04 first."
assert "device"                in globals(), "Run Block 01 first."

# Reproducibility
RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

# The edge type we are predicting
TARGET_EDGE_TRIPLET = ("drug", "indication", "disease")  # PyG-style triplet key
TARGET_EDGE_LABEL   = "indication"

# Split ratios
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15
assert abs(TRAIN_RATIO + VAL_RATIO + TEST_RATIO - 1.0) < 1e-9, "Ratios must sum to 1."

# Negative sampling config
NEG_RATIO = 1


# A. separate target vs context edges

split_df    = edges_model_ready_df.copy()
target_df   = split_df[split_df["edge_label"] == TARGET_EDGE_LABEL].copy()
context_df  = split_df[split_df["edge_label"] != TARGET_EDGE_LABEL].copy()

# Confirm all target edges are drug → disease
bad_target = target_df[
    (target_df["src_type"] != "drug") | (target_df["dst_type"] != "disease")
]
if not bad_target.empty:
    raise ValueError(f"Non-drug→disease rows inside indication edges:\n{bad_target.head()}")

# Deduplicate target edges on the (drug, disease) pair
target_df = target_df.drop_duplicates(subset=["src_uid", "dst_uid"]).copy()
target_df.reset_index(drop=True, inplace=True)

print("=" * 55)
print("  EDGE SPLIT OVERVIEW")
print("=" * 55)
print(f"  Target edges ({TARGET_EDGE_LABEL}) : {len(target_df):,}")
print(f"  Context edges                       : {len(context_df):,}")


# B. train / val / test split on target edges

# First split: train vs (val + test)
train_df, temp_df = train_test_split(
    target_df, test_size=(1.0 - TRAIN_RATIO),
    random_state=RANDOM_STATE, shuffle=True,
)

# Second split: val vs test
val_relative = VAL_RATIO / (VAL_RATIO + TEST_RATIO)
val_df, test_df = train_test_split(
    temp_df, test_size=(1.0 - val_relative),
    random_state=RANDOM_STATE, shuffle=True,
)

# Tag each split for traceability
train_df = train_df.copy(); train_df["split"] = "train"
val_df   = val_df.copy();   val_df["split"]   = "val"
test_df  = test_df.copy();  test_df["split"]  = "test"

# Full target split table (useful for leakage checks and evidence extraction)
target_split_df = pd.concat([train_df, val_df, test_df], ignore_index=True)

# Verify no pair appears in two splits
train_pairs = set(zip(train_df["src_uid"], train_df["dst_uid"]))
val_pairs   = set(zip(val_df["src_uid"],   val_df["dst_uid"]))
test_pairs  = set(zip(test_df["src_uid"],  test_df["dst_uid"]))
assert len(train_pairs & val_pairs)  == 0, "Train/val leakage!"
assert len(train_pairs & test_pairs) == 0, "Train/test leakage!"
assert len(val_pairs   & test_pairs) == 0, "Val/test leakage!"

print(f"\n  Train indication edges : {len(train_df):,}")
print(f"  Val   indication edges : {len(val_df):,}")
print(f"  Test  indication edges : {len(test_df):,}")


# C. negative sampler

def sample_negatives(
    num_src:    int,
    num_dst:    int,
    forbidden:  set,
    n_samples:  int,
    seed:       int = 42,
) -> torch.Tensor:
    """
    Sample n_samples drug–disease pairs that are NOT in forbidden.

    Uses vectorized batch sampling with rejection for efficiency:
      - Draw 3×n candidates at once (oversample)
      - Filter out forbidden pairs
      - Repeat until we have enough

    Returns LongTensor of shape [2, n_samples].
    """
    rng = np.random.default_rng(seed)
    collected = set()

    while len(collected) < n_samples:
        need  = n_samples - len(collected)
        srcs  = rng.integers(0, num_src, need * 3).tolist()
        dsts  = rng.integers(0, num_dst, need * 3).tolist()
        for s, d in zip(srcs, dsts):
            pair = (s, d)
            if pair not in forbidden and pair not in collected:
                collected.add(pair)
                if len(collected) >= n_samples:
                    break

    pairs = list(collected)
    return torch.tensor(pairs, dtype=torch.long).t().contiguous()


# Build the full set of known positives
all_positive_pairs = set(
    zip(target_split_df["src_index"].tolist(),
        target_split_df["dst_index"].tolist())
)

num_drugs    = num_nodes_dict["drug"]
num_diseases = num_nodes_dict["disease"]


# D. build supervision tensors

def df_to_edge_index(df: pd.DataFrame) -> torch.Tensor:
    """Convert a split DataFrame to a LongTensor of shape [2, E]."""
    if df.empty:
        return torch.empty((2, 0), dtype=torch.long)
    arr = df[["src_index", "dst_index"]].to_numpy().T
    return torch.tensor(arr, dtype=torch.long)

train_pos_edge_label_index = df_to_edge_index(train_df)
val_pos_edge_label_index   = df_to_edge_index(val_df)
test_pos_edge_label_index  = df_to_edge_index(test_df)

# Pre-sample val and test negatives
val_neg_edge_label_index = sample_negatives(
    num_src=num_drugs, num_dst=num_diseases,
    forbidden=all_positive_pairs,
    n_samples=val_pos_edge_label_index.size(1) * NEG_RATIO,
    seed=RANDOM_STATE,
)
test_neg_edge_label_index = sample_negatives(
    num_src=num_drugs, num_dst=num_diseases,
    forbidden=all_positive_pairs,
    n_samples=test_pos_edge_label_index.size(1) * NEG_RATIO,
    seed=RANDOM_STATE + 1,
)

print(f"\n  Supervision tensor shapes:")
print(f"    train_pos : {tuple(train_pos_edge_label_index.shape)}")
print(f"    val_pos   : {tuple(val_pos_edge_label_index.shape)}")
print(f"    val_neg   : {tuple(val_neg_edge_label_index.shape)}")
print(f"    test_pos  : {tuple(test_pos_edge_label_index.shape)}")
print(f"    test_neg  : {tuple(test_neg_edge_label_index.shape)}")



# E. build pyg heterodata

train_data = HeteroData()

# Node stores — only num_nodes, no features
for node_type, n_nodes in num_nodes_dict.items():
    train_data[node_type].num_nodes = n_nodes

# Build the edge index dict for the TRAINING graph:
#   - All context edges (full set)
#   - Only TRAIN indication edges (val/test are held out)
train_graph_edge_index_dict = {}

# Context edges
for triplet, group in context_df.groupby(["src_type", "edge_label", "dst_type"]):
    src_t, rel, dst_t = triplet
    ei = torch.tensor(group[["src_index", "dst_index"]].to_numpy().T, dtype=torch.long)
    train_graph_edge_index_dict[(src_t, rel, dst_t)] = ei

# Training indication edges only
train_target_ei = df_to_edge_index(train_df)
train_graph_edge_index_dict[TARGET_EDGE_TRIPLET] = train_target_ei

# Assign to HeteroData
for edge_triplet, edge_index in train_graph_edge_index_dict.items():
    src_t, rel, dst_t = edge_triplet
    train_data[(src_t, rel, dst_t)].edge_index = edge_index

# Validation
bad_shapes = [
    (et, tuple(train_data[et].edge_index.shape))
    for et in train_data.edge_types
    if train_data[et].edge_index.dim() != 2 or train_data[et].edge_index.shape[0] != 2
]
if bad_shapes:
    raise ValueError(f"Invalid edge_index shapes: {bad_shapes}")

print("\n" + "=" * 55)
print("  PyG HETERODATA OBJECT")
print("=" * 55)
print(train_data)
print()
print("  Node types  :", train_data.node_types)
print("  Edge types  :", len(train_data.edge_types))
print()
print("  Node counts:")
for nt in train_data.node_types:
    print(f"    {nt}: {train_data[nt].num_nodes:,}")
print()
print("  Edge counts:")
for et in train_data.edge_types:
    print(f"    {et}: {train_data[et].edge_index.shape[1]:,}")

print()
print("=" * 55)
print("  BLOCK 06 COMPLETE — objects available:")
print("    train_data")
print("    train_pos_edge_label_index")
print("    val_pos_edge_label_index  / val_neg_edge_label_index")
print("    test_pos_edge_label_index / test_neg_edge_label_index")
print("    all_positive_pairs")
print("    TARGET_EDGE_TRIPLET")
print("    target_split_df")
print("    sample_negatives(...)")
print("=" * 55)

  EDGE SPLIT OVERVIEW
  Target edges (indication) : 9,388
  Context edges                       : 778,105

  Train indication edges : 6,571
  Val   indication edges : 1,408
  Test  indication edges : 1,409

  Supervision tensor shapes:
    train_pos : (2, 6571)
    val_pos   : (2, 1408)
    val_neg   : (2, 1408)
    test_pos  : (2, 1409)
    test_neg  : (2, 1409)

  PyG HETERODATA OBJECT
HeteroData(
  disease={ num_nodes=17080 },
  drug={ num_nodes=6681 },
  effect={ num_nodes=990 },
  phenotype={ num_nodes=15311 },
  protein={ num_nodes=19059 },
  (disease, disease_associated_with_protein, protein)={ edge_index=[2, 80411] },
  (disease, disease_has_phenotype, phenotype)={ edge_index=[2, 150317] },
  (disease, disease_parent_child_disease, disease)={ edge_index=[2, 64388] },
  (drug, contraindication, disease)={ edge_index=[2, 30675] },
  (drug, drug_carrier_protein, protein)={ edge_index=[2, 864] },
  (drug, drug_enzyme_protein, protein)={ edge_index=[2, 5317] },
  (drug, drug_has_sid